# Target Recon Toolkit

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:

!pip install python-whois

import pandas as pd
import requests
import re
import socket
import whois
from urllib.parse import quote
from IPython.display import display, Markdown

# -----------------------
# 1. Basic WHOIS lookup
# -----------------------
def get_whois_info(domain):
    try:
        w = whois.whois(domain)
        return {
            "domain": domain,
            "registrar": w.registrar,
            "creation_date": str(w.creation_date),
            "expiration_date": str(w.expiration_date),
            "emails": w.emails,
            "name_servers": w.name_servers
        }
    except Exception as e:
        return {"domain": domain, "error": str(e)}

# -----------------------
# 2. crt.sh - Find subdomains
# -----------------------
def get_subdomains_crtsh(domain):
    url = f"https://crt.sh/?q=%25.{domain}&output=json"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            subdomains = set(entry["name_value"] for entry in data)
            flat_subs = set()
            for entry in subdomains:
                flat_subs.update(entry.split('\n'))
            return list(flat_subs)
    except Exception as e:
        print(f"[crt.sh] Error: {e}")
    return []

# -----------------------
# 3. Basic GeoIP Lookup
# -----------------------
def geoip_lookup(ip):
    try:
        response = requests.get(f"http://ip-api.com/json/{ip}").json()
        return {
            "IP": ip,
            "Country": response.get("country"),
            "City": response.get("city"),
            "ISP": response.get("isp"),
            "ASN": response.get("as")
        }
    except Exception as e:
        return {"IP": ip, "error": str(e)}

# -----------------------
# 4. Email Breach Check (mock check)
# -----------------------
def check_email_breach(email):
    try:
        domain = email.split("@")[-1]
        if domain in ["gmail.com", "yahoo.com"]:
            return {"email": email, "status": "High risk: common phishing target"}
        return {"email": email, "status": "No known issues"}
    except:
        return {"email": email, "status": "Invalid format"}

# -----------------------
# 5. Reverse DNS lookup
# -----------------------
def reverse_dns(ip):
    try:
        return socket.gethostbyaddr(ip)[0]
    except:
        return "N/A"

# -----------------------
# Master Recon Function
# -----------------------
def recon_target(domain_or_ip_or_email):
    display(Markdown(f"## 🔍 Recon Report for `{domain_or_ip_or_email}`"))

    result_data = {}

    if re.match(r"[^@]+@[^@]+\.[^@]+", domain_or_ip_or_email):  # Email
        result_data["Breach_Check"] = check_email_breach(domain_or_ip_or_email)

    elif re.match(r"\d{1,3}(\.\d{1,3}){3}", domain_or_ip_or_email):  # IP
        result_data["GeoIP"] = geoip_lookup(domain_or_ip_or_email)
        result_data["Reverse_DNS"] = reverse_dns(domain_or_ip_or_email)

    else:  # Domain
        result_data["WHOIS"] = get_whois_info(domain_or_ip_or_email)
        result_data["Subdomains"] = get_subdomains_crtsh(domain_or_ip_or_email)

    for section, data in result_data.items():
        display(Markdown(f"### 🔹 {section}"))
        if isinstance(data, list):
            df = pd.DataFrame(data, columns=["Subdomain"])
        elif isinstance(data, dict):
            df = pd.DataFrame([data])
        else:
            df = pd.DataFrame(data)
        display(df)

# -----------------------
# Example Use
# -----------------------

#  Run this cell to test on a domain or IP or email
target = "example.com"  # 🔄 Change to any domain, IP, or email
recon_target(target)
